# v15 — Full v3 FE + Optuna LightGBM (GPU)

Tuned LightGBM with full feature engineering pipeline.  
Time-aware validation (last 15% by timestamp), 50 Optuna trials, GPU acceleration where available.

**Expected runtime:** ~15-25 min on T4 GPU.  
**Output:** `submission.csv`

In [ ]:
%matplotlib inline
import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
from sklearn.metrics import r2_score
from sklearn.model_selection import KFold
from lightgbm import LGBMRegressor, early_stopping as lgb_early_stop
import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)
import pygeohash as pgh

pd.set_option('display.max_columns', 80)
np.random.seed(42)

def _detect_lgb_gpu():
    try:
        X = np.array([[1.0, 2.0], [3.0, 4.0], [5.0, 6.0], [7.0, 8.0]], dtype=np.float32)
        y = np.array([1.0, 2.0, 3.0, 4.0], dtype=np.float32)
        LGBMRegressor(n_estimators=2, device='gpu', max_bin=63, verbose=-1).fit(X, y)
        return True
    except Exception as e:
        print(f'  LightGBM GPU not available: {type(e).__name__}: {str(e)[:80]}')
        return False

LGB_GPU = _detect_lgb_gpu()
print(f'LightGBM GPU: {LGB_GPU}')

def lgb_params(base):
    out = dict(base)
    if LGB_GPU:
        out['device'] = 'gpu'
        out['max_bin'] = 63
    else:
        out['n_jobs'] = -1
    return out

## Load Data

In [ ]:
train = pd.read_csv('train.csv')
test  = pd.read_csv('test.csv')
print(f'Train: {train.shape}  Test: {test.shape}')

## Feature Engineering

In [ ]:
class FeatureEngineer:
    @staticmethod
    def _decode_gh(gh):
        try:
            lat, lng, lat_err, lng_err = pgh.decode_exactly(gh)
            return float(lat), float(lng), float(lat_err), float(lng_err)
        except Exception:
            return 0.0, 0.0, 0.0, 0.0
    @staticmethod
    def _parse_ts(ts):
        h, m = ts.split(':')
        return int(h), int(m)
    @staticmethod
    def _time_of_day(h):
        if   h <= 5:  return 0
        elif h <= 11: return 1
        elif h <= 17: return 2
        else:         return 3

    def fit_transform(self, df):
        df = df.copy()
        self._temp_med_global = float(df['Temperature'].median())
        self._temp_med_gh = df.groupby('geohash')['Temperature'].median()
        self._rt_mode_global = (df['RoadType'].mode().iloc[0]
                                if len(df['RoadType'].mode()) else 'Residential')
        self._rt_mode_gh = df.groupby('geohash')['RoadType'].apply(
            lambda x: x.mode().iloc[0] if len(x.mode()) else self._rt_mode_global)
        self._wx_mode_global = (df['Weather'].mode().iloc[0]
                                if len(df['Weather'].mode()) else 'Sunny')
        self._wx_mode_gh = df.groupby('geohash')['Weather'].apply(
            lambda x: x.mode().iloc[0] if len(x.mode()) else self._wx_mode_global)
        df = self._impute(df)

        self._max_day = int(df['day'].max())
        self._rt_cols = sorted(df['RoadType'].dropna().unique().tolist())
        self._wx_cols = sorted(df['Weather'].dropna().unique().tolist())

        gb = df.groupby('geohash')['demand']
        self._gh_std   = gb.std().fillna(0)
        self._gh_p25   = gb.quantile(0.25)
        self._gh_p75   = gb.quantile(0.75)
        self._gh_max   = gb.max()
        self._gh_count = gb.count().astype(float)
        self._gh_mean  = gb.mean()
        self._g_mean   = float(df['demand'].mean())
        self._g_std    = float(df['demand'].std())

        ts_tmp = df['timestamp'].apply(self._parse_ts)
        df['_hour'] = ts_tmp.apply(lambda x: x[0])
        df['_tod']  = df['_hour'].apply(self._time_of_day)
        self._hour_mean = df.groupby('_hour')['demand'].mean()
        self._hour_std  = df.groupby('_hour')['demand'].std().fillna(0)
        self._tod_mean  = df.groupby('_tod')['demand'].mean()
        df = df.drop(columns=['_hour', '_tod'])

        gh_mean_d = self._gh_mean.to_dict()
        self._neighbor_mean_d = {}
        for gh in df['geohash'].unique():
            try:
                nbrs = list(pgh.neighbors(gh).values())
                self._neighbor_mean_d[gh] = float(np.mean(
                    [gh_mean_d.get(n, self._g_mean) for n in nbrs]))
            except Exception:
                self._neighbor_mean_d[gh] = self._g_mean

        gh4 = df['geohash'].str[:4]; gh5 = df['geohash'].str[:5]
        self._gh4_labels = {v: i for i, v in enumerate(sorted(gh4.unique()))}
        self._gh5_labels = {v: i for i, v in enumerate(sorted(gh5.unique()))}

        _, self._temp_bin_edges = pd.cut(df['Temperature'], bins=5, retbins=True)
        self._temp_bin_edges[0]  = -np.inf
        self._temp_bin_edges[-1] =  np.inf

        self._gh_rank = self._gh_mean.rank(pct=True).to_dict()
        self.fitted = True
        return self._transform(df)

    def transform(self, df):
        df = df.copy(); df = self._impute(df); return self._transform(df)

    def _impute(self, df):
        df['Temperature'] = df['Temperature'].fillna(
            df['geohash'].map(self._temp_med_gh).fillna(self._temp_med_global))
        df['RoadType'] = df['RoadType'].fillna(
            df['geohash'].map(self._rt_mode_gh).fillna(self._rt_mode_global))
        df['Weather'] = df['Weather'].fillna(
            df['geohash'].map(self._wx_mode_gh).fillna(self._wx_mode_global))
        return df

    def _transform(self, df):
        ts = df['timestamp'].apply(self._parse_ts)
        df['hour']         = ts.apply(lambda x: x[0])
        df['minute']       = ts.apply(lambda x: x[1])
        df['hour_sin']     = np.sin(2 * np.pi * df['hour'] / 24)
        df['hour_cos']     = np.cos(2 * np.pi * df['hour'] / 24)
        df['minute_sin']   = np.sin(2 * np.pi * df['minute'] / 60)
        df['minute_cos']   = np.cos(2 * np.pi * df['minute'] / 60)
        df['is_rush_hour'] = df['hour'].isin([7,8,9,17,18,19]).astype(int)
        df['time_of_day']  = df['hour'].apply(self._time_of_day)

        geo = df['geohash'].apply(self._decode_gh)
        df['geohash_lat']     = geo.apply(lambda x: x[0])
        df['geohash_lng']     = geo.apply(lambda x: x[1])
        df['geohash_lat_err'] = geo.apply(lambda x: x[2])
        df['geohash_lng_err'] = geo.apply(lambda x: x[3])

        df['day_sin']   = np.sin(2 * np.pi * df['day'] / self._max_day)
        df['day_cos']   = np.cos(2 * np.pi * df['day'] / self._max_day)
        df['day_mod_7'] = df['day'] % 7

        df['LargeVehicles'] = (df['LargeVehicles'] == 'Allowed').astype(int)
        df['Landmarks']     = (df['Landmarks']     == 'Yes').astype(int)
        for rt in self._rt_cols:
            df[f'RoadType_{rt}'] = (df['RoadType'] == rt).astype(int)
        for wx in self._wx_cols:
            df[f'Weather_{wx}']  = (df['Weather']  == wx).astype(int)

        df['geohash_std_demand'] = df['geohash'].map(self._gh_std).fillna(self._g_std)
        df['geohash_p25_demand'] = df['geohash'].map(self._gh_p25).fillna(self._g_mean*0.4)
        df['geohash_p75_demand'] = df['geohash'].map(self._gh_p75).fillna(self._g_mean*1.6)
        df['geohash_max_demand'] = df['geohash'].map(self._gh_max).fillna(self._g_mean*2.5)
        df['geohash_count']      = df['geohash'].map(self._gh_count).fillna(1.0)
        df['neighbor_mean_demand']= df['geohash'].map(self._neighbor_mean_d).fillna(self._g_mean)

        df['hour_mean_demand'] = df['hour'].map(self._hour_mean).fillna(self._g_mean)
        df['hour_std_demand']  = df['hour'].map(self._hour_std).fillna(self._g_std)
        df['tod_mean_demand']  = df['time_of_day'].map(self._tod_mean).fillna(self._g_mean)

        df['geohash_4'] = df['geohash'].str[:4]
        df['geohash_5'] = df['geohash'].str[:5]
        df['geohash_4_label'] = df['geohash_4'].map(self._gh4_labels).fillna(-1).astype(int)
        df['geohash_5_label'] = df['geohash_5'].map(self._gh5_labels).fillna(-1).astype(int)

        df['temp_binned'] = pd.cut(df['Temperature'], bins=self._temp_bin_edges,
                                    labels=False, include_lowest=True).fillna(2).astype(int)
        df['demand_rank_in_geohash'] = df['geohash'].map(self._gh_rank).fillna(0.5)

        rt_num = {'Highway': 3, 'Street': 2, 'Residential': 1}
        df['RoadType_numeric'] = df['RoadType'].map(rt_num).fillna(0).astype(float)
        df['lanes_x_roadtype'] = df['NumberofLanes'] * df['RoadType_numeric']

        wx_num = {'Sunny': 1, 'Cloudy': 2, 'Rainy': 3, 'Foggy': 4, 'Snowy': 5}
        df['Weather_numeric'] = df['Weather'].map(wx_num).fillna(1).astype(float)
        df['temp_x_weather']  = df['Temperature'] * df['Weather_numeric']
        df['hour_x_geohash_lat'] = df['hour'] * df['geohash_lat']
        return df

fe = FeatureEngineer()
train_fe = fe.fit_transform(train)
test_fe  = fe.transform(test)
print(f'Train FE: {train_fe.shape}  Test FE: {test_fe.shape}')

## OOF Target Encoding + Post-OOF Derived Features

In [ ]:
def oof_encode(train_df, test_df, group_cols, col_name, alpha=10, n_splits=5):
    if isinstance(group_cols, str): group_cols = [group_cols]
    gm = float(train_df['demand'].mean())
    n = len(train_df)
    sort_key = (train_df['day']*1440 + train_df['hour']*60 + train_df['minute']).values
    sort_pos = np.argsort(sort_key); unsort_pos = np.argsort(sort_pos)
    df_s = train_df.iloc[sort_pos].reset_index(drop=True)
    enc = np.full(n, gm, dtype=float)
    for tr_idx, va_idx in KFold(n_splits=n_splits, shuffle=False).split(np.arange(n)):
        stats = df_s.iloc[tr_idx].groupby(group_cols)['demand'].agg(['mean','count'])
        stats['s'] = (stats['count']*stats['mean'] + alpha*gm) / (stats['count']+alpha)
        d = stats['s'].to_dict()
        if len(group_cols) == 1:
            keys = df_s.iloc[va_idx][group_cols[0]].tolist()
        else:
            keys = list(zip(*[df_s.iloc[va_idx][c].tolist() for c in group_cols]))
        enc[va_idx] = [d.get(k, gm) for k in keys]
    train_df[col_name] = enc[unsort_pos]
    full = train_df.groupby(group_cols)['demand'].agg(['mean','count'])
    full['s'] = (full['count']*full['mean'] + alpha*gm) / (full['count']+alpha)
    fd = full['s'].to_dict()
    if len(group_cols) == 1:
        tk = test_df[group_cols[0]].tolist()
    else:
        tk = list(zip(*[test_df[c].tolist() for c in group_cols]))
    test_df[col_name] = [fd.get(k, gm) for k in tk]

print('Applying OOF encodings...')
oof_encode(train_fe, test_fe, 'geohash',                 'geohash_mean_demand');           print('  1/6')
oof_encode(train_fe, test_fe, ['geohash','hour'],         'geohash_hour_mean_demand');      print('  2/6')
oof_encode(train_fe, test_fe, ['geohash','time_of_day'],  'geohash_timeofday_mean_demand'); print('  3/6')
oof_encode(train_fe, test_fe, 'geohash_4',                'geohash_4_mean_demand');         print('  4/6')
oof_encode(train_fe, test_fe, 'geohash_5',                'geohash_5_mean_demand');         print('  5/6')
oof_encode(train_fe, test_fe, 'temp_binned',              'temp_binned_mean_demand');       print('  6/6')

for df in [train_fe, test_fe]:
    df['location_time']     = df['geohash_mean_demand']      * df['hour_sin']
    df['demand_vs_hour']    = df['geohash_mean_demand']      / (df['hour_mean_demand'] + 1e-9)
    df['gh_hour_vs_global'] = df['geohash_hour_mean_demand'] / (df['hour_mean_demand'] + 1e-9)

print(f'Final train_fe: {train_fe.shape}  test_fe: {test_fe.shape}')

## Feature arrays + time-aware split

In [ ]:
EXCLUDE = {'Index', 'geohash', 'timestamp', 'demand', 'RoadType', 'Weather',
           'geohash_4', 'geohash_5'}
feature_cols = [c for c in train_fe.columns if c not in EXCLUDE]
print(f'Total features: {len(feature_cols)}')

X_all = train_fe[feature_cols].values.astype(np.float32)
y_all = train_fe['demand'].values.astype(np.float32)
X_test = test_fe[feature_cols].values.astype(np.float32)
for arr in (X_all, X_test):
    if np.isnan(arr).any():
        arr[:] = np.nan_to_num(arr, nan=0.0)

sort_keys = (train_fe['day']*1440 + train_fe['hour']*60 + train_fe['minute']).values
sorted_idx = np.argsort(sort_keys)
split = int(len(sorted_idx) * 0.85)
tr_i = sorted_idx[:split]; va_i = sorted_idx[split:]
X_tr, y_tr = X_all[tr_i], y_all[tr_i]
X_va, y_va = X_all[va_i], y_all[va_i]
print(f'X_all: {X_all.shape}  Train: {X_tr.shape}  Val: {X_va.shape}')

## Optuna Tuning — 50 trials, GPU LightGBM

In [ ]:
def objective(trial):
    params = dict(
        n_estimators      = 2000,
        learning_rate     = trial.suggest_float('learning_rate', 0.01, 0.15, log=True),
        num_leaves        = trial.suggest_int('num_leaves', 63, 511),
        min_child_samples = trial.suggest_int('min_child_samples', 5, 100),
        subsample         = trial.suggest_float('subsample', 0.5, 1.0),
        subsample_freq    = 1,
        colsample_bytree  = trial.suggest_float('colsample_bytree', 0.5, 1.0),
        reg_alpha         = trial.suggest_float('reg_alpha', 1e-3, 10.0, log=True),
        reg_lambda        = trial.suggest_float('reg_lambda', 1e-3, 10.0, log=True),
        random_state=42, verbose=-1,
    )
    m = LGBMRegressor(**lgb_params(params))
    m.fit(X_tr, y_tr, eval_set=[(X_va, y_va)],
          callbacks=[lgb_early_stop(50, verbose=False)])
    return r2_score(y_va, m.predict(X_va))

study = optuna.create_study(direction='maximize',
                             sampler=optuna.samplers.TPESampler(seed=42))
study.optimize(objective, n_trials=50, show_progress_bar=True)

print(f'\nBest val R² = {study.best_value:.4f}')
print('Best params:')
for k, v in study.best_params.items():
    print(f'  {k}: {v}')

## Retrain Tuned LightGBM on Full Train + Submit

In [ ]:
best_params = study.best_params.copy()
best_params['n_estimators'] = 2000
best_params['random_state'] = 42
best_params['verbose'] = -1
best_params['subsample_freq'] = 1

check_model = LGBMRegressor(**lgb_params(best_params))
check_model.fit(X_tr, y_tr, eval_set=[(X_va, y_va)],
                 callbacks=[lgb_early_stop(50, verbose=False)])
best_iter = check_model.best_iteration_
final_iters = int(best_iter * 1.15) + 1
print(f'best_iter = {best_iter}  final_iters = {final_iters}')

final_params = dict(best_params); final_params['n_estimators'] = final_iters
final_model = LGBMRegressor(**lgb_params(final_params))
final_model.fit(X_all, y_all)

final_preds = final_model.predict(X_test)
final_preds = np.clip(final_preds, 0.0, 1.0)

sub = pd.DataFrame({'Index': test['Index'].values, 'demand': final_preds})
assert sub.shape == (41778, 2)
assert sub['demand'].isna().sum() == 0
sub.to_csv('submission.csv', index=False)

print(f'\nsubmission.csv saved  |  shape: {sub.shape}')
print(f'min={final_preds.min():.4f} max={final_preds.max():.4f} '
      f'mean={final_preds.mean():.4f} std={final_preds.std():.4f}')
print(f'\nBest val R² during tuning: {study.best_value:.4f}')